<a href="https://colab.research.google.com/github/SanjaraT/Deep-Leraning-with-PyTorch/blob/main/Simple_Q_A_system_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import numpy as np


In [23]:
df = pd.read_csv('/content/drive/MyDrive/100_Unique_QA_Dataset.csv')

In [24]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [12]:
# tokenization
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [13]:
# dictionary for words
vocab = {'<UNK>':0}

In [25]:
def add_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)

In [28]:
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [30]:
import torch
from torch.utils.data import Dataset, DataLoader

In [31]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [32]:
dataset = QADataset(df, vocab)

In [33]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [34]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[  1,   2,   3,  92, 137,  19,   3,  45]]) tensor([185])
tensor([[10, 75, 76]]) tensor([77])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([259])
tensor([[ 10, 140,   3, 141, 270,  93, 271,   5,   3, 272]]) tensor([273])
tensor([[10, 29,  3, 30, 31]]) tensor([32])
tensor([[  1,   2,   3,   4,   5, 236, 237]]) tensor([238])
tensor([[ 42, 137,   2, 138,  39, 175, 269]]) tensor([99])
tensor([[  1,   2,   3, 146,  86,  19, 192, 193]]) tensor([194])
tensor([[  1,   2,   3, 180, 181, 182, 183]]) tensor([184])
tensor([[  1,   2,   3, 234,   5, 235]]) tensor([131])
tensor([[  1,   2,   3,  37, 133,   5,  26]]) tensor([134])
tensor([[  1,   2,   3,   4,   5, 286]]) tensor([287])
tensor([[  1,   2,   3, 122, 123,  19,   3,  45]]) tensor([124])
tensor([[ 10,  75,   3, 296,  19, 297]]) tensor([298])
tensor([[ 42, 200,   2,  14, 201, 202, 203, 204]]) tensor([205])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([246])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([16])
tensor([[ 10,

In [35]:
import torch.nn as nn
# RNN model
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [36]:
learning_rate = 0.001
epochs = 20

In [37]:
model = SimpleRNN(len(vocab))

In [38]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [39]:
# training pipeline

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 519.424113
Epoch: 2, Loss: 445.169795
Epoch: 3, Loss: 365.987783
Epoch: 4, Loss: 307.269967
Epoch: 5, Loss: 257.850017
Epoch: 6, Loss: 211.297871
Epoch: 7, Loss: 168.499349
Epoch: 8, Loss: 131.814727
Epoch: 9, Loss: 102.142942
Epoch: 10, Loss: 78.701454
Epoch: 11, Loss: 60.709473
Epoch: 12, Loss: 47.682174
Epoch: 13, Loss: 38.164644
Epoch: 14, Loss: 31.430691
Epoch: 15, Loss: 25.669198
Epoch: 16, Loss: 21.551168
Epoch: 17, Loss: 18.165264
Epoch: 18, Loss: 15.500968
Epoch: 19, Loss: 13.372299
Epoch: 20, Loss: 11.700311


In [47]:
# prediction
def predict(model, question, threshold=0.5):

  #  question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)
  output = model(question_tensor)

  #logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

In [48]:
predict(model, "Who is the author of '1984'?")

In [49]:
predict(model, "What is the capital of Bangladesh?")

I don't know
